In [9]:
#### IMPORTS ####
from datetime import datetime
import os
import pandas as pd

#### CONSTANTS  (column names from the PSX file, kept in one place) ####
COL_ISIN = "ISIN"
COL_SYMBOL = "SYMBOL"
COL_COMPANY = "COMPANY"
COL_PRICE = "PRICE"
COL_WEIGHT = "IDX WT %"

#### MODULE 1: Ask for the date and build the CSV file name ####
def get_csv_filename_from_user():
    """
    Ask the user for the date of the constituents file, build the matching CSV name (2026-06-10 -> 20260610_kse100.csv), and loop until the date is valid AND the file exists on disk.
    Returns BOTH the file name and the date object (the date is reused later to name the output files).
    """
    while True:
        user_text = input(
            "Enter the date of the constituents CSV (YYYY-MM-DD), "
            "e.g. 2026-06-10: "
        ).strip()

        try:
            # Convert the text into a real date; reject bad input.
            valid_date = datetime.strptime(user_text, "%Y-%m-%d")
        except ValueError:
            print("  >> Invalid date. Use YYYY-MM-DD with a real date.\n")
            continue  # go back to the top of the loop and ask again.
        # Build the expected input file name, e.g. 20260610_kse100.csv
        file_name = valid_date.strftime("%Y%m%d") + "_kse100.csv"

        # Only proceed if that file actually exists in this folder.
        if os.path.exists(file_name):
            return file_name, valid_date

        print(f"  >> File '{file_name}' not found in this folder. \n ")

#### MODULE 2: Load the CSV into a pandas table ####

def load_csv(file_name):
    """
    Read the CSV into a DataFrame and strip stray spaces from column names. Returns the DataFrame, or None if reading fails.
    """
    try:
        data_table = pd.read_csv(file_name)

        # Clean column names like " PRICE " -> "PRICE".
        data_table.columns = data_table.columns.str.strip()

        print(f"Loaded '{file_name}': {len(data_table)} companies.\n")
        return data_table

    except Exception as error:
        print(f"  >> Could not read the CSV file: {error}")
        return None

#### MODULE 3: Ask for the basket size (total value in PKR) ####

def get_basket_size_from_user():
    """
    Ask for the total basket value in rupees without any comma. Loops until a positive number is entered. Returns the value as a float.
    """
    while True:
        cleaned_text = input(
            "Enter the basket size in PKR (e.g. 50,000,000): "
        ).strip()

        try:
            # float() converts text to a number; fails on bad input.
            basket_size = float(cleaned_text)

        except ValueError:
            print("  >> That is not a number. Please try again.\n")
            continue

        # The basket must be worth something positive.
        if basket_size > 0:
            return basket_size

        print("  >> Basket size must be greater than zero.\n")


#### MODULE 4: Ask for the intended cash percentage ####

def get_cash_percent_from_user():
    """
    Ask what percentage of the basket should be held as cash. Must be between 0 and 100 (not including 100, because a basket of 100% cash holds no shares at all). Returns a float.
    """
    while True:
        user_text = input(
            "Enter the intended cash % of the basket (e.g. 2): "
        ).strip()

        try:
            cash_percent = float(user_text)
        except ValueError:
            print("  >> That is not a number. Please try again.\n")
            continue

        # Valid range check: 0 is allowed, 100 is not.
        if 0 <= cash_percent < 100:
            return cash_percent

        print("  >> Cash % must be at least 0 and below 100.\n")

#### MODULE 5: Build the shared part of every output name ####

def build_output_name_base(date_object, basket_size):
    """
    Build the text that all three output file names share:
        yyyymmdd_kse100_..._<basketsize>
    Program returns both parts; each save-module combines them with its own middle word ('basket', 'summary', 'dropped').
    """
    # The date with no dashes, e.g. 20260610.
    date_part = date_object.strftime("%Y%m%d")

    # The basket size as plain digits, no decimals, no commas.
    size_part = f"{basket_size:.0f}"

    return date_part, size_part

#### MODULE 6: Calculate exact and rounded shares for every company ####

def calculate_shares(data_table, basket_size, cash_percent):

    """
    Add three new columns to the table:
      EXACT SHARES   - the (fractional) ideal number of shares,
      BASKET SHARES  - exact shares rounded to the NEAREST whole share,
      SHARE VALUE    - basket shares x price (rupee value held).

    The equity target is the part of the basket NOT meant to be cash:
        equity target = basket size x (100 - cash%) / 100
    Returns the modified table and the equity target.
    """
    # The rupee amount intended to be invest in shares (rest is cash).
    equity_target = basket_size * (100.0 - cash_percent) / 100.0

    # Each company's target rupee value = its index weight share of  the equity target. (Weight is in %, so divide by 100.)
    target_value = data_table[COL_WEIGHT] / 100.0 * equity_target


    # Ideal fractional shares = target value / share price.
    data_table["EXACT SHARES"] = target_value / data_table[COL_PRICE]

    # round(0) rounds each value to the nearest whole number (e.g. 12.4 -> 12, 12.6 -> 13). astype(int) stores it as a whole number instead of 12.0 / 13.0.
    data_table["BASKET SHARES"] = (
        data_table["EXACT SHARES"].round(0).astype(int)
    )

    # The rupee value actually held in each company after rounding.
    data_table["SHARE VALUE"] = (
        data_table["BASKET SHARES"] * data_table[COL_PRICE]
    )

    return data_table, equity_target

#### MODULE 7: Repair the basket if rounding made cash negative ####

def repair_negative_cash(data_table, basket_size):
    """
    RULE: cash = basket size - total share value, and it must be >= 0.
    Rounding UP many names can make total share value exceed the basket size (negative cash).
    Repair method:
      - Look at every company's rounding OVERSHOOT in rupees:
            overshoot = (basket shares - exact shares) x price
            A positive overshoot means rounding gave it MORE value than its exact target.
      - Remove ONE share from the company with the LARGEST overshoot (it frees the most cash while removing the worst overweight).
      - Repeat until cash is no longer negative.
    Returns the table and the number of trims performed.
    """
    trims_done = 0  # counter, just for reporting.

    # Loop until the cash condition is satisfied.
    while True:
        # Current cash position implied by the holdings.
        cash = basket_size - data_table["SHARE VALUE"].sum()

        # If cash is already fine, stop repairing.
        if cash >= 0:
            break

        # Rupee overshoot of every company (positive = rounded up).
        overshoot = (
            (data_table["BASKET SHARES"] - data_table["EXACT SHARES"])
            * data_table[COL_PRICE]
        )

        # We can only trim companies that still hold at least 1 share.
        # Where shares are 0, force the overshoot to a very low value i.e., negative infinity using float("-inf")
        # so idxmax() below never selects them.
        overshoot[data_table["BASKET SHARES"] <= 0] = float("-inf")

        # idxmax() returns the ROW LABEL of the largest overshoot.
        worst_row = overshoot.idxmax()

        # Trim one share from that company.
        data_table.loc[worst_row, "BASKET SHARES"] -= 1

        # Update that company's rupee value after the trim.
        data_table.loc[worst_row, "SHARE VALUE"] = (
            data_table.loc[worst_row, "BASKET SHARES"]
            * data_table.loc[worst_row, COL_PRICE]
        )

        trims_done += 1  # record that a repair trim happened.

    return data_table, trims_done


#### MODULE 8: Compute basket weights and weight differences ####

def calculate_basket_weights(data_table):
    """
    Add two columns:
      BASKET WT %  - company's share value as % of TOTAL EQUITY VALUE
                     (cash excluded),
      WT DIFF      - IDX WT % minus BASKET WT %, rounded to 2 decimals
                     (positive = the basket is UNDERWEIGHT that name).
    Then sort all companies by index weight, largest first.
    Returns the sorted table.
    """
    # Total rupee value of all shares actually held (the denominator).
    total_equity_value = data_table["SHARE VALUE"].sum()

    # Each company's realized weight inside the equity portion.
    data_table["BASKET WT %"] = (
        data_table["SHARE VALUE"] / total_equity_value * 100.0
    ).round(2)

    # Difference vs the index weight, to 2 decimal places.
    data_table["WT DIFF"] = (
        data_table[COL_WEIGHT] - data_table["BASKET WT %"]
    ).round(2)

    # Sort by index weight, biggest companies on top.
    # reset_index(drop=True) renumbers rows 0,1,2,... after sorting.
    data_table = data_table.sort_values(
        by=COL_WEIGHT, ascending=False
    ).reset_index(drop=True)

    return data_table

#### MODULE 9: Build the summary as a small table ####

def build_summary_table(basket_size, cash_percent, equity_target,
                        data_table, trims_done):
    """
    Compute the headline numbers of the finished basket ONCE and put  them into a tiny two-column table (ITEM, VALUE).
    """
    # Value of all shares actually held, after rounding and repairs.
    equity_actual = data_table["SHARE VALUE"].sum()

    # Cash = whatever part of the basket size is not in shares.
    cash_actual = basket_size - equity_actual

    # Actual cash as a percentage of the basket size.
    cash_actual_percent = cash_actual / basket_size * 100.0

    # Count of companies that ended up with zero shares.
    dropped_count = (data_table["BASKET SHARES"] == 0).sum()

    # Build the summary as a list of (item, value) pairs. Values are formatted as text here so both the screen and the CSV show the same friendly numbers (e.g. 50,000,000.00).
    summary_rows = [
        ("Basket size (PKR)",      f"{basket_size:,.2f}"),
        ("Intended cash %",        f"{cash_percent:.2f}"),
        ("Equity target (PKR)",    f"{equity_target:,.2f}"),
        ("Equity actual (PKR)",    f"{equity_actual:,.2f}"),
        ("Cash actual (PKR)",      f"{cash_actual:,.2f}"),
        ("Cash actual %",          f"{cash_actual_percent:.2f}"),
        ("Repair trims performed", f"{trims_done}"),
        ("Companies with 0 shares", f"{dropped_count}"),
    ]

    # Convert the pairs into a 2-column DataFrame named ITEM / VALUE.
    summary_table = pd.DataFrame(summary_rows, columns=["ITEM", "VALUE"])

    return summary_table


#### MODULE 10: Print the summary from the summary table ####

def report_summary(summary_table):
    """
    Print the summary table to the screen, one item per line. (The numbers to be prepared by build_summary_table.)
    """
    print("BASKET SUMMARY")
    print("-" * 60)

    # itertuples() loops through the table row by row.
    for row in summary_table.itertuples():
        # row.ITEM and row.VALUE are the two columns of each row.
        # '<26' pads the item name to 26 characters so values align (overall left aligned).
        print(f"  {row.ITEM:<26}: {row.VALUE}")
    print()


#### MODULE 11: Build and print the 0-share (dropped) table ####

def build_dropped_table(data_table):
    """
    Make a mini-table of the companies whose rounded share count is ZERO (they fail to make the basket). Returns the mini-table - it may be EMPTY: an empty CSV with just the header row still documents that nothing was dropped.
    """
    # Boolean mask: True for rows holding zero shares.
    zero_mask = data_table["BASKET SHARES"] == 0

    # Keep only the dropped rows and the columns useful for analysis.
    dropped_table = data_table.loc[
        zero_mask,
        [COL_ISIN, COL_SYMBOL, COL_COMPANY, COL_PRICE, COL_WEIGHT,
         "EXACT SHARES"]
    ].copy()  

    # .copy() makes it independent of the big table.

    return dropped_table


def report_dropped_companies(dropped_table):
    """
    Print the dropped-companies table to the screen.
    """
    print("COMPANIES THAT FAIL TO MAKE THE BASKET (0 shares)")
    print("-" * 60)

    # len() of a table = its number of rows.
    if len(dropped_table) == 0:
        print("  None - every constituent received at least 1 share.\n")
        return

    print(dropped_table.to_string(index=False))
    print(f"\n  Total companies dropped: {len(dropped_table)}")

    # add a general tip for the reader
    print("  Tip: a LARGER basket size lets more small names in.\n")

#### MODULE 12: Print the full constituents table ####

def report_full_table(data_table):
    """
    Print ALL companies (including the 0-share ones), already sorted by index weight descending, with the basket weight and weight difference.
    """
    print("FULL BASKET TABLE (sorted by index weight, largest first)")
    print("-" * 60)

    # Choose and order the columns for display.
    display_columns = [
        COL_SYMBOL, COL_COMPANY, COL_PRICE, COL_WEIGHT,
        "BASKET SHARES", "SHARE VALUE", "BASKET WT %", "WT DIFF",
    ]

    # to_string(index=False) prints the whole table without row numbers.
    print(data_table[display_columns].to_string(index=False))
    print()

#### MODULE 13: Save all THREE output CSVs ####

def save_all_outputs(data_table, summary_table, dropped_table,
                     date_object, basket_size):
    """
    Save the three result files, every name carrying the basket size:
        <date>_kse100_basket_<size>.csv    - full constituents table
        <date>_kse100_summary_<size>.csv   - headline numbers
        <date>_kse100_dropped_<size>.csv   - 0-share companies
    Prints each saved name. Returns True if all three saved fine.
    """
    # Get the two shared name parts (date digits, size digits).
    date_part, size_part = build_output_name_base(date_object, basket_size)

    # Columns to keep in the FULL table file, in a sensible order.
    basket_columns = [
        COL_ISIN, COL_SYMBOL, COL_COMPANY, COL_PRICE, COL_WEIGHT,
        "EXACT SHARES", "BASKET SHARES", "SHARE VALUE",
        "BASKET WT %", "WT DIFF",
    ]

    # Describe the three save jobs as (file name, table to save) pairs,
    # then looping over them - so the try/except logic is written ONCE.
    save_jobs = [
        (f"{date_part}_kse100_basket_{size_part}.csv",
         data_table[basket_columns]),
        (f"{date_part}_kse100_summary_{size_part}.csv",
         summary_table),
        (f"{date_part}_kse100_dropped_{size_part}.csv",
         dropped_table),
    ]

    all_saved = True  # assuming success; flip to False on any failure.

    for file_name, table in save_jobs:
        try:
            # index=False: do not write pandas' internal row numbers.
            table.to_csv(file_name, index=False)
            print(f"Saved: {file_name}")
        except Exception as error:
            print(f"  >> Could not save '{file_name}': {error}")
            all_saved = False

    return all_saved



#### MODULE 14 (MAIN): connect all the modules in order ####

def main():
    """
    The conductor:
      1. Locate and load the constituents CSV.
      2. Ask for basket size and intended cash %.
      3. Compute rounded shares per company.
      4. Repair the basket if cash went negative.
      5. Compute basket weights and differences.
      6. Build the summary and dropped tables (used for both printing and saving - computed once, used twice).
      7. Report everything on screen.
      8. Save the three CSVs, named with date AND basket size.
    """
    print("=" * 60)
    print("  KSE-100 Creation Basket Builder")
    print("=" * 60)

    # STEP 1: find and load the input data.
    file_name, chosen_date = get_csv_filename_from_user()
    data_table = load_csv(file_name)
    if data_table is None:
        print("Stopping: the constituents CSV could not be loaded.")
        return

    # STEP 2: get the two basket parameters from the user.
    basket_size = get_basket_size_from_user()
    cash_percent = get_cash_percent_from_user()
    print()  # blank line, just for tidy output.

    # STEP 3: compute exact and rounded shares for every company.
    data_table, equity_target = calculate_shares(
        data_table, basket_size, cash_percent
    )

    # STEP 4: enforce the "cash never negative" rule.
    data_table, trims_done = repair_negative_cash(data_table, basket_size)

    # STEP 5: realized basket weights and differences, sorted.
    data_table = calculate_basket_weights(data_table)

    # STEP 6: build the two extra tables (summary + dropped).
    summary_table = build_summary_table(
        basket_size, cash_percent, equity_target, data_table, trims_done
    )
    dropped_table = build_dropped_table(data_table)

    # STEP 7: all the screen reports, in a logical reading order.
    report_summary(summary_table)
    report_dropped_companies(dropped_table)
    report_full_table(data_table)

    # STEP 8: save all three CSVs with size-tagged names.
    save_all_outputs(data_table, summary_table, dropped_table,
                     chosen_date, basket_size)

    print("\nAll done!")


## PROGRAM ENTRY POINT (main() runs only when this file is executed directly, not when its functions are imported by another script) ##

if __name__ == "__main__":
    main()


  KSE-100 Creation Basket Builder


Enter the date of the constituents CSV (YYYY-MM-DD), e.g. 2026-06-10:  2025-10-08


Loaded '20251008_kse100.csv': 100 companies.



Enter the basket size in PKR (e.g. 50,000,000):  300000
Enter the intended cash % of the basket (e.g. 2):  2



BASKET SUMMARY
------------------------------------------------------------
  Basket size (PKR)         : 300,000.00
  Intended cash %           : 2.00
  Equity target (PKR)       : 294,000.00
  Equity actual (PKR)       : 291,398.84
  Cash actual (PKR)         : 8,601.16
  Cash actual %             : 2.87
  Repair trims performed    : 0
  Companies with 0 shares   : 5

COMPANIES THAT FAIL TO MAKE THE BASKET (0 shares)
------------------------------------------------------------
        ISIN SYMBOL                               COMPANY    PRICE  IDX WT %  EXACT SHARES
PK0025101012 NESTLE               Nestle Pakistan Limited  8349.98  0.384137      0.135253
PK0076901013   UPFL       Unilever Pakistan Foods Limited 30000.00  0.189773      0.018598
PK0032001015   RMPL Rafhan Maize Products Company Limited  9724.82  0.182240      0.055095
PK0068201018   SSOM                 S.S.Oil Mills Limited   383.50  0.010733      0.082280
PK0051701016   BNWM           Bannu Woollen Mills Limited   